# F-06: REddyProc-style met-driver gap-filling — does it beat mean-imputation? (RFm)

Every model so far **mean-imputes** the meteorological drivers (e.g. SWIN ~52-75% present). `src/data/reddyproc_pipeline.py`
gap-fills them properly (linear interp + mean-diurnal-course, preserving the diurnal cycle) and adds GPP/Reco from
nighttime Lloyd-Taylor partitioning (D-33). This A/B/C tests the **F-05 best config** (partial pool + density + lags +
pruned management) varying ONLY the driver preprocessing:

- **imputed** — raw met drivers + mean-imputation (current baseline)
- **metfill** — REddyProc-gap-filled met drivers (`__f` columns) + lags on the filled SWC/TS
- **metfill+gpp** — + GPP & Reco features

Clean A/B: identical training rows (target-valid) + identical gap scenarios (fixed seed). Per-tower R². Towers 2/4/9.

In [1]:
from pathlib import Path
import datetime
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
HOURLY=Path("../../data/Hourly"); RESULTS=Path("../../results")
PLAUS_LOW,PLAUS_HIGH=-500,3000
N_REPS,MASK_FRAC=5,0.25
SCENARIOS={"vs":1,"s":4,"m":32,"l":288,"m1":"mixed"}
LSU={"cattle":1.0,"sheep":0.1,"lamb":0.05}
AUX=["_hour_sin","_hour_cos","_doy_sin","_doy_cos"]
LAG_HOURS=[168,336,504,672]
CATCHMENT_AREA_HA={1:4.81,2:6.65,3:6.62,4:7.75,5:6.47,6:3.86,7:2.60,8:7.02,9:7.75,10:1.82,11:1.76,12:1.78,13:1.75,14:1.72,15:1.54}
TOWERS=[2,4,9]

## 1  Configs + load (consolidated + FCO2 + management + reddyproc_processed)

In [2]:
C4="Catchment 4 After  2013/08/13"
def cfg(t,cat,catstr):
    M={k:v for k,v in {
        "sw":f"SWIN_1_1_1 [Tower {t}]","ta":f"TA_0_0_1 [Tower {t}]","vpd":f"VPD_0_0_1 [Tower {t}]",
        "ppfd":f"PPFD_1_1_1 [Tower {t}]","rn":f"RN_1_1_1 [Tower {t}]","ws":f"WS_0_0_1 [Tower {t}]",
        "ustar":f"USTAR_0_0_1 [Tower {t}]","shf":f"SHF_1_1_1 [Tower {t}]",
        "precip":f"Precipitation (mm) [{catstr}]","swc":f"Soil Moisture @ 10cm Depth (%) [{catstr}]",
        "ts":"TS_1_1_1 [Tower 9]"}.items()}
    return {"tower":t,"area_ha":CATCHMENT_AREA_HA[cat],
        "target":f"FCH4_1_1_1 [Tower {t}]","ssitc":f"FCH4_SSITC_TEST_1_1_1 [Tower {t}]",
        "met":M,"fc":f"FC_1_1_1 [Tower {t}]",
        "gpp":f"GPP [Tower {t}]","reco":f"Reco [Tower {t}]",
        "livestock":{sp:f"{sp}_{catstr}" for sp in LSU},
        "mgmt_cut":f"mgmt_t{t}_cut_recency","mgmt_manure":f"mgmt_t{t}_manure_recency",
        "train_yrs":[2018] if t==2 else list(range(2018,2022)),
        "test_yrs":[2019] if t==2 else list(range(2022,2024))}
TOWER_CONFIGS={2:cfg(2,2,"Catchment 2"),4:cfg(4,4,C4),9:cfg(9,9,"Catchment 9")}
df=pd.read_csv(HOURLY/"consolidated_hourly.csv",low_memory=False)
df["Datetime"]=pd.to_datetime(df["Datetime"],format="mixed"); df=df.set_index("Datetime")
for extra in ["fco2_gapfilled.csv","management_features.csv","reddyproc_processed.csv"]:
    e=pd.read_csv(HOURLY/extra,low_memory=False); e["Datetime"]=pd.to_datetime(e["Datetime"],format="mixed")
    df=df.join(e.set_index("Datetime"),how="left")
for _t in TOWERS:
    c=f"FC_1_1_1 [Tower {_t}]"
    if c in df.columns: df[c]=df[f"FC_gapfilled [Tower {_t}]"]
df_raw=df; print("loaded",df_raw.shape)

loaded (70153, 522)


## 2  Functions

In [3]:
def insert_calendar_gaps(d,target,test_yrs,gh,n_reps=N_REPS,seed=0):
    tm=d.index.year.isin(test_yrs); tt=d.index[tm]; va=d.loc[tm,target].notna().values
    n=len(tt); tn=max(1,int(va.sum()*MASK_FRAC)); rb=np.random.default_rng(seed); reps=[]
    for _ in range(n_reps):
        rng=np.random.default_rng(int(rb.integers(0,2**31))); occ=np.zeros(n,bool); m=0
        for sp in rng.permutation(n):
            if m>=tn: break
            g=int(rng.choice([1,4,32,288])) if gh=="mixed" else gh; ep=min(int(sp)+g,n)
            if occ[sp:ep].any(): continue
            occ[sp:ep]=True; m+=int(va[sp:ep].sum())
        reps.append(tt[occ & va])
    return reps

def generic_frame(t, filled, add_gpp):
    c=TOWER_CONFIGS[t]; d=df_raw.copy(); tgt=c["target"]
    d.loc[~d[c["ssitc"]].isin([0,1]),tgt]=np.nan
    d.loc[d[tgt].notna() & ~d[tgt].between(PLAUS_LOW,PLAUS_HIGH,inclusive="both"),tgt]=np.nan
    h,doy=d.index.hour,d.index.dayofyear
    d["_hour_sin"]=np.sin(2*np.pi*h/24); d["_hour_cos"]=np.cos(2*np.pi*h/24)
    d["_doy_sin"]=np.sin(2*np.pi*doy/365); d["_doy_cos"]=np.cos(2*np.pi*doy/365)
    M=c["met"]
    def src(k):
        col=M[k]; fcol=col+"__f"
        return d[fcol] if (filled and fcol in d.columns) else d[col]
    liv=c["livestock"]; area=c["area_ha"]
    lsu=sum(d[liv[s]].fillna(0)*w for s,w in LSU.items())
    g=pd.DataFrame(index=d.index); g["target"]=d[tgt]
    metkeys=["sw","ta","vpd","ppfd","rn","ws","ustar","shf","precip","swc","ts"]
    for k in metkeys: g[k]=src(k)
    g["fc"]=d[c["fc"]]
    for a in AUX: g[a]=d[a]
    g["lsu_dens"]=lsu/area; g["graze"]=(sum(d[liv[s]].fillna(0) for s in LSU)>0).astype(float)
    swc_s, ts_s = src("swc"), src("ts")
    for lag in LAG_HOURS:
        g[f"swc_lag{lag}"]=swc_s.shift(lag); g[f"ts_lag{lag}"]=ts_s.shift(lag)
    g["mgmt_cut"]=d[c["mgmt_cut"]]; g["mgmt_manure"]=d[c["mgmt_manure"]]
    if add_gpp:
        g["gpp"]=d[c["gpp"]]; g["reco"]=d[c["reco"]]
    for tt in TOWERS: g[f"is_t{tt}"]=1.0 if tt==t else 0.0
    g["_year"]=d.index.year
    return g

METK=["sw","ta","vpd","ppfd","rn","ws","ustar","shf","precip","swc","ts"]
LAGC=[f"swc_lag{l}" for l in LAG_HOURS]+[f"ts_lag{l}" for l in LAG_HOURS]
DUM=[f"is_t{t}" for t in TOWERS]
def featlist(add_gpp):
    f=METK+["fc"]+AUX+["lsu_dens","graze"]+LAGC+["mgmt_cut","mgmt_manure"]+DUM
    return f+(["gpp","reco"] if add_gpp else [])

def fit(feat,train_df):
    imp=SimpleImputer(strategy="mean")
    rf=RandomForestRegressor(n_estimators=500,min_samples_leaf=5,n_jobs=-1,random_state=42)
    rf.fit(imp.fit_transform(train_df[feat].values),train_df["target"].values); return rf,imp
def eval_on(rf,imp,feat,g,c):
    recs=[]; rg={sc:insert_calendar_gaps(g,"target",c["test_yrs"],gh) for sc,gh in SCENARIOS.items()}
    for sc,reps in rg.items():
        for rep,ts in enumerate(reps):
            if len(ts)<5: continue
            yt=g.loc[ts,"target"].values; yp=rf.predict(imp.transform(g.loc[ts,feat].values))
            recs.append({"scenario":sc,"rep":rep,"R2":r2_score(yt,yp)})
    return pd.DataFrame(recs)

## 3  Train (pooled) + evaluate the three variants

In [4]:
VARIANTS={"imputed":(False,False),"metfill":(True,False),"metfill+gpp":(True,True)}
rows=[]
for name,(filled,add_gpp) in VARIANTS.items():
    feat=featlist(add_gpp)
    frames={t:generic_frame(t,filled,add_gpp) for t in TOWERS}
    # identical training rows across variants: target-valid in train years (no met dropna)
    parts=[]
    for t in TOWERS:
        gg=frames[t]; yrs=TOWER_CONFIGS[t]["train_yrs"]
        parts.append(gg[gg["_year"].isin(yrs) & gg["target"].notna()])
    pool=pd.concat(parts,ignore_index=True)
    rf,imp=fit(feat,pool)
    for t in TOWERS:
        m=eval_on(rf,imp,feat,frames[t],TOWER_CONFIGS[t]); m["tower"]=f"Tower {t}"; m["variant"]=name
        rows.append(m)
    print(f"{name:14s} trained on {len(pool):,} rows, {len(feat)} feats")
res=pd.concat(rows,ignore_index=True); print("eval rows:",len(res))

imputed        trained on 21,049 rows, 31 feats


metfill        trained on 21,049 rows, 31 feats


metfill+gpp    trained on 21,049 rows, 33 feats
eval rows: 225


## 4  Results — does met gap-filling beat mean-imputation?

In [5]:
VAR=["imputed","metfill","metfill+gpp"]
for t in TOWERS:
    sub=res[res.tower==f"Tower {t}"]
    tbl=sub.groupby(["variant","scenario"])["R2"].median().unstack("scenario").reindex(index=VAR).reindex(columns=list(SCENARIOS)).round(3)
    print(f"\n=== Tower {t} ==="); print(tbl.to_string())
print("\n--- overall median R2 + deltas vs imputed ---")
for t in TOWERS:
    s=res[res.tower==f"Tower {t}"].groupby("variant")["R2"].median()
    print(f"Tower {t}: imputed {s['imputed']:+.3f} | metfill {s['metfill']:+.3f} (d {s['metfill']-s['imputed']:+.3f}) | +gpp {s['metfill+gpp']:+.3f} (d {s['metfill+gpp']-s['imputed']:+.3f})")


=== Tower 2 ===
scenario        vs      s      m      l     m1
variant                                       
imputed     -0.154 -0.114 -0.135 -0.155 -0.120
metfill     -0.039  0.031 -0.050 -0.080 -0.012
metfill+gpp -0.037  0.046 -0.045 -0.079 -0.008

=== Tower 4 ===
scenario        vs      s      m      l     m1
variant                                       
imputed      0.232  0.150  0.227  0.060 -0.109
metfill      0.223  0.160  0.242  0.059 -0.108
metfill+gpp  0.250  0.187  0.301  0.081 -0.098

=== Tower 9 ===
scenario        vs      s      m      l     m1
variant                                       
imputed      0.302  0.267  0.325  0.192  0.265
metfill      0.323  0.298  0.338  0.203  0.282
metfill+gpp  0.356  0.316  0.374  0.269  0.332

--- overall median R2 + deltas vs imputed ---
Tower 2: imputed -0.126 | metfill -0.050 (d +0.076) | +gpp -0.045 (d +0.081)
Tower 4: imputed +0.116 | metfill +0.138 (d +0.022) | +gpp +0.163 (d +0.047)
Tower 9: imputed +0.270 | metfill +0.287 (d

## 5  Append to benchmarks.csv

In [6]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="F-06"]
out=[]
for r in res.to_dict("records"):
    out.append({"replication":"F-06","model":"RFm_"+r["variant"].replace("+","_"),"tower":r["tower"],
        "feature_set":r["variant"],"scenario":r["scenario"],"rep":r["rep"],"split":"pool_T2+4+9",
        "R2":round(r["R2"],4),"date":today,"notes":"F06 REddyProc-style met gap-fill vs mean-impute; +GPP/Reco (D-33)"})
new=pd.DataFrame(out); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} F-06 rows. Total {len(comb)}.")

Wrote 225 F-06 rows. Total 2740.
